In [39]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


In [40]:
df = pd.read_csv('../../data/cleaned_df.csv')
df = df.drop_duplicates()
df = df[df['StateOrProvince'] == 'CA']
df.head()

/var/folders/5g/sd7vmfvs2rn86tg601yfsjx80000gn/T/ipykernel_49727/3836542674.py:1: DtypeWarning: Columns (0: PoolPrivateYN, 1: FireplaceYN) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../data/cleaned_df.csv')


,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,ListingContractDate,StateOrProvince,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude
0,1650000.0,3452.0,11255.0,4.0,2.0,2.0,5.0,19,NaN,24059 Regents Park Circle,...,2025-02-09,CA,True,True,True,True,False,3.0,-118.558321,34.404533
1,500000.0,2961.0,8258.0,3.0,2.0,1.0,5.0,25,NaN,11592 Greene Court,...,2025-02-11,CA,False,True,True,True,False,2.0,-117.410510,34.522797
2,1650000.0,2929.0,12907.0,3.0,2.0,1.0,5.0,20,"Tile,Wood",2628 Rudy Street,...,2025-03-03,CA,True,False,True,True,False,3.0,-117.866868,33.968692
3,770000.0,1532.0,3300.0,2.0,2.0,2.0,3.0,17,NaN,10602 Porto Court,...,2024-12-27,CA,False,False,False,True,NaN,2.0,-117.102586,32.825896
4,885000.0,2130.0,8100.0,3.0,1.0,4.0,4.0,8,"Carpet,Vinyl",10420 Oneida Avenue,...,2025-03-03,CA,False,False,False,False,False,2.0,-118.426100,34.260374


In [41]:
import geopandas as gpd

districts = gpd.read_file('../../data/CA_district_areas.geojson')
districts = districts.to_crs("EPSG:4326")
districts.head()

,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictName,DistrictType,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAreaSqMi,LocaleCode,LocaleDesc,geometry
0,1,2025-26,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1302,12.1,4259,39.5,11.248886,21,"21 - Suburban, Large","MULTIPOLYGON (((-122.22678 37.72651, -122.2267..."
1,2,2025-26,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,363,9.7,1247,33.3,1.789975,21,"21 - Suburban, Large","POLYGON ((-122.28671 37.89852, -122.28673 37.8..."
2,3,2025-26,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,0,0.0,1118,11.9,2710,28.8,10.434281,12,"12 - City, Midsize","POLYGON ((-122.25606 37.89834, -122.25607 37.8..."
3,4,2025-26,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,2,0.0,1186,12.2,3784,39.0,66.885261,21,"21 - Suburban, Large","MULTIPOLYGON (((-122.01375 37.64265, -122.0114..."
4,5,2025-26,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,0,0.0,98,16.1,407,66.7,1.273923,21,"21 - Suburban, Large","POLYGON ((-122.29663 37.8311, -122.29778 37.83..."


In [42]:
district_info = districts[['DistrictName', 'DistrictType', 'geometry']].copy()
district_info.head()

,DistrictName,DistrictType,geometry
0,Alameda Unified,Unified,"MULTIPOLYGON (((-122.22678 37.72651, -122.2267..."
1,Albany City Unified,Unified,"POLYGON ((-122.28671 37.89852, -122.28673 37.8..."
2,Berkeley Unified,Unified,"POLYGON ((-122.25606 37.89834, -122.25607 37.8..."
3,Castro Valley Unified,Unified,"MULTIPOLYGON (((-122.01375 37.64265, -122.0114..."
4,Emery Unified,Unified,"POLYGON ((-122.29663 37.8311, -122.29778 37.83..."


In [43]:
geo_df = gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']), crs='EPSG:4326')
geo_df.head()

,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,StateOrProvince,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude,geometry
0,1650000.0,3452.0,11255.0,4.0,2.0,2.0,5.0,19,NaN,24059 Regents Park Circle,...,CA,True,True,True,True,False,3.0,-118.558321,34.404533,POINT (-118.55832 34.40453)
1,500000.0,2961.0,8258.0,3.0,2.0,1.0,5.0,25,NaN,11592 Greene Court,...,CA,False,True,True,True,False,2.0,-117.410510,34.522797,POINT (-117.41051 34.5228)
2,1650000.0,2929.0,12907.0,3.0,2.0,1.0,5.0,20,"Tile,Wood",2628 Rudy Street,...,CA,True,False,True,True,False,3.0,-117.866868,33.968692,POINT (-117.86687 33.96869)
3,770000.0,1532.0,3300.0,2.0,2.0,2.0,3.0,17,NaN,10602 Porto Court,...,CA,False,False,False,True,NaN,2.0,-117.102586,32.825896,POINT (-117.10259 32.8259)
4,885000.0,2130.0,8100.0,3.0,1.0,4.0,4.0,8,"Carpet,Vinyl",10420 Oneida Avenue,...,CA,False,False,False,False,False,2.0,-118.426100,34.260374,POINT (-118.4261 34.26037)


In [44]:
df_districts = gpd.sjoin(geo_df, district_info, how="left", predicate="within").drop(columns=['index_right', 'geometry'])
df_districts.head()

,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude,DistrictName,DistrictType
0,1650000.0,3452.0,11255.0,4.0,2.0,2.0,5.0,19,NaN,24059 Regents Park Circle,...,True,True,True,True,False,3.0,-118.558321,34.404533,Newhall,Elementary
0,1650000.0,3452.0,11255.0,4.0,2.0,2.0,5.0,19,NaN,24059 Regents Park Circle,...,True,True,True,True,False,3.0,-118.558321,34.404533,William S. Hart Union High,High
1,500000.0,2961.0,8258.0,3.0,2.0,1.0,5.0,25,NaN,11592 Greene Court,...,False,True,True,True,False,2.0,-117.410510,34.522797,Adelanto Elementary,Elementary
1,500000.0,2961.0,8258.0,3.0,2.0,1.0,5.0,25,NaN,11592 Greene Court,...,False,True,True,True,False,2.0,-117.410510,34.522797,Victor Valley Union High,High
2,1650000.0,2929.0,12907.0,3.0,2.0,1.0,5.0,20,"Tile,Wood",2628 Rudy Street,...,True,False,True,True,False,3.0,-117.866868,33.968692,Rowland Unified,Unified


In [45]:
district_features = (df_districts.reset_index().pivot_table(index='index', columns='DistrictType', values='DistrictName', aggfunc='first'))
district_features

DistrictType,Elementary,High,Unified
index,,,
0,Newhall,William S. Hart Union High,NaN
1,Adelanto Elementary,Victor Valley Union High,NaN
2,NaN,NaN,Rowland Unified
3,NaN,NaN,San Diego Unified
4,NaN,NaN,Los Angeles Unified
...,...,...,...
80497,NaN,NaN,Yucaipa-Calimesa Joint Unified
80498,NaN,NaN,Orange Unified
80499,Fullerton Elementary,Fullerton Joint Union High,NaN


In [46]:
district_split = district_features.rename(columns={'Elementary': 'ElementaryDistrict', 'High': 'HighDistrict', 'Unified': 'UnifiedDistrict'})
district_split

DistrictType,ElementaryDistrict,HighDistrict,UnifiedDistrict
index,,,
0,Newhall,William S. Hart Union High,NaN
1,Adelanto Elementary,Victor Valley Union High,NaN
2,NaN,NaN,Rowland Unified
3,NaN,NaN,San Diego Unified
4,NaN,NaN,Los Angeles Unified
...,...,...,...
80497,NaN,NaN,Yucaipa-Calimesa Joint Unified
80498,NaN,NaN,Orange Unified
80499,Fullerton Elementary,Fullerton Joint Union High,NaN


In [48]:
df = df_districts.join(district_split).drop(columns=['DistrictType', 'DistrictName', 'ElementaryDistrict', 'HighDistrict']).drop_duplicates()
df

,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,StateOrProvince,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude,UnifiedDistrict
0,1650000.0,3452.0,11255.0,4.0,2.0,2.0,5.0,19,NaN,24059 Regents Park Circle,...,CA,True,True,True,True,False,3.0,-118.558321,34.404533,NaN
1,500000.0,2961.0,8258.0,3.0,2.0,1.0,5.0,25,NaN,11592 Greene Court,...,CA,False,True,True,True,False,2.0,-117.410510,34.522797,NaN
2,1650000.0,2929.0,12907.0,3.0,2.0,1.0,5.0,20,"Tile,Wood",2628 Rudy Street,...,CA,True,False,True,True,False,3.0,-117.866868,33.968692,Rowland Unified
3,770000.0,1532.0,3300.0,2.0,2.0,2.0,3.0,17,NaN,10602 Porto Court,...,CA,False,False,False,True,NaN,2.0,-117.102586,32.825896,San Diego Unified
4,885000.0,2130.0,8100.0,3.0,1.0,4.0,4.0,8,"Carpet,Vinyl",10420 Oneida Avenue,...,CA,False,False,False,False,False,2.0,-118.426100,34.260374,Los Angeles Unified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80497,594900.0,1867.0,10032.0,3.0,1.0,0.0,4.0,63,NaN,12995 Leith Way,...,CA,True,True,True,True,False,2.0,-117.036902,34.019226,Yucaipa-Calimesa Joint Unified
80498,1705000.0,2776.0,6000.0,3.0,2.0,1.0,4.0,34,"Carpet,Wood",7741 E Briarwood Road,...,CA,False,True,True,True,False,3.0,-117.770601,33.788646,Orange Unified
80499,741000.0,1404.0,6408.0,2.0,1.0,3.0,3.0,4,NaN,2620 W Olive Avenue,...,CA,False,False,False,False,False,2.0,-117.973674,33.862109,NaN
80500,985000.0,1748.0,7310.0,2.0,1.0,3.0,3.0,122,NaN,10505 Halbrent,...,CA,True,True,False,True,False,2.0,-118.466216,34.261109,Los Angeles Unified


In [49]:
df['UnifiedDistrict'].isna().sum()/len(df)

np.float64(0.243692468229419)

In [18]:
model_df = df.copy()
# Building to lot ratio
model_df["LotLivingRatio"] = np.where(df["LotSizeSquareFeet"] > 0, df["LivingArea"] / df["LotSizeSquareFeet"], np.nan)
# Bedroom to bathroom ratio
model_df['BathroomBedroomRatio'] = np.where(df['BedroomsTotal'] > 0, df['BathroomsTotalInteger']/df['BedroomsTotal'], np.nan)
# Total amount of house amenities
model_df['TotalAmenityCount'] = df['ViewYN'] + df['PoolPrivateYN'] + df['AttachedGarageYN'] + df['FireplaceYN']
# Total number of different types of flooring
model_df['FlooringCount'] = df['Flooring'].str.split(',').str.len()

In [19]:
model_df.columns

Index(['ClosePrice', 'LivingArea', 'LotSizeSquareFeet',
       'BathroomsTotalInteger', 'Stories', 'MainLevelBedrooms',
       'BedroomsTotal', 'DaysOnMarket', 'Flooring', 'UnparsedAddress',
       'AssociationFeeFrequency', 'MLSAreaMajor', 'ElementarySchool',
       'SubdivisionName', 'City', 'PurchaseContractDate',
       'MiddleOrJuniorSchool', 'HighSchool', 'HighSchoolDistrict', 'Levels',
       'ListingKey', 'CloseDate', 'PropertyType', 'ListingKeyNumeric',
       'CountyOrParish', 'MlsStatus', 'PropertySubType', 'ListingId',
       'ContractStatusChangeDate', 'ListingContractDate', 'StateOrProvince',
       'ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN',
       'NewConstructionYN', 'ParkingTotal', 'Longitude', 'Latitude',
       'UnifiedDistrict', 'LotLivingRatio', 'BathroomBedroomRatio',
       'TotalAmenityCount', 'FlooringCount'],
      dtype='str')

In [28]:
target = 'ClosePrice'

# Categorize columns

cat_col = ['Flooring', 'AssociationFeeFrequency', 
        'MLSAreaMajor', 'ElementarySchool', 'SubdivisionName', 'City', 
        'PurchaseContractDate', 'MiddleOrJuniorSchool', 'HighSchool',
        'HighSchoolDistrict', 'Levels', 'ListingKey', 'CloseDate', 
        'PropertyType', 'ListingKeyNumeric', 'CountyOrParish',
        'PropertySubType', 'ListingId', 'ContractStatusChangeDate',
        'ListingContractDate', 'StateOrProvince', 'UnifiedDistrict']

bool_col = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']

num_col = ['LivingArea', 'LotSizeSquareFeet', 'BathroomsTotalInteger', 
            'Stories', 'MainLevelBedrooms', 'BedroomsTotal', 'DaysOnMarket',
            'LotLivingRatio', 'BathroomBedroomRatio', 'TotalAmenityCount', 'FlooringCount']

required_cols = ['ParkingTotal', 'Longitude', 'Latitude']


# Keep only columns that exist in the dataframe
num_col = [col for col in num_col if col in model_df.columns]
cat_col = [col for col in cat_col if col in model_df.columns]
bool_col = [col for col in bool_col if col in model_df.columns]

keep_cols = [target] + cat_col + bool_col + required_cols + num_col 
model_df = model_df[keep_cols]
# model_df = model_df.drop_duplicates()

model_df.head()


,ClosePrice,Flooring,AssociationFeeFrequency,MLSAreaMajor,ElementarySchool,SubdivisionName,City,PurchaseContractDate,MiddleOrJuniorSchool,HighSchool,...,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,LotLivingRatio,BathroomBedroomRatio,TotalAmenityCount,FlooringCount
0,1650000.0,NaN,Monthly,VSUM - Valencia Summit,Meadows,NaN,Valencia,2025-03-13,Placerita,Hart,...,11255.0,4.0,2.0,2.0,5.0,19,0.306708,0.800000,4,NaN
1,500000.0,NaN,NaN,ADL - Adelanto,NaN,NaN,Adelanto,2025-03-28,NaN,NaN,...,8258.0,3.0,2.0,1.0,5.0,25,0.358561,0.600000,3,NaN
2,1650000.0,"Tile,Wood",Quarterly,652 - Rowland Heights,Ybarra,NaN,Rowland Heights,2025-03-27,NaN,Rowland,...,12907.0,3.0,2.0,1.0,5.0,20,0.226931,0.600000,3,2.0
3,770000.0,NaN,Monthly,92124 - Tierrasanta,NaN,NaN,San Diego,2025-04-07,NaN,NaN,...,3300.0,2.0,2.0,2.0,3.0,17,0.464242,0.666667,1,NaN
4,885000.0,"Carpet,Vinyl",NaN,PAC - Pacoima,NaN,NaN,Pacoima,2025-03-11,NaN,NaN,...,8100.0,3.0,1.0,4.0,4.0,8,0.262963,0.750000,0,2.0


### Model Building

In [29]:

def preproc_df(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    # ----------------------------
    # Drop rows with missing values in key columns
    # ----------------------------
    required_cols = ['ParkingTotal', 'Longitude', 'Latitude']
    train_df = train_df.dropna(subset=required_cols)
    test_df = test_df.dropna(subset=required_cols)

    # Remove invalid values
    train_df = train_df[
        (train_df["ClosePrice"] > 0) &
        (train_df["LivingArea"] > 0) &
        (train_df["BathroomsTotalInteger"] > 0) &
        (train_df["DaysOnMarket"] > 0)
    ]

    test_df = test_df[
        (test_df["ClosePrice"] > 0) &
        (test_df["LivingArea"] > 0) &
        (test_df["BathroomsTotalInteger"] > 0) &
        (test_df["DaysOnMarket"] > 0)
    ]

    # ----------------------------
    # Missing value handling
    # ----------------------------

    for col in cat_col:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna("Unknown")
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna("Unknown")

    # Missing indicator creation
    for col in train_df.columns:
        if train_df[col].isna().sum() > 0:
            train_df[f"{col}_was_missing"] = train_df[col].isna().astype(int) #
            test_df[f"{col}_was_missing"] = test_df[col].isna().astype(int) #

    if "YearBuilt" in train_df.columns:
        median = train_df["YearBuilt"].median()
        train_df["YearBuilt"] = train_df["YearBuilt"].fillna(median)
        test_df["YearBuilt"] = test_df["YearBuilt"].fillna(median)

    if "StreetNumberNumeric" in train_df.columns:
        train_df["StreetNumberNumeric"] = train_df["StreetNumberNumeric"].fillna(-1)
        test_df["StreetNumberNumeric"] = test_df["StreetNumberNumeric"].fillna(-1)

    train_df[bool_col] = train_df[bool_col].fillna(False)
    test_df[bool_col] = test_df[bool_col].fillna(False)

    for col in num_col:
        if col in train_df.columns:
            median = train_df[col].median()
            train_df[col] = train_df[col].fillna(median)
            test_df[col] = test_df[col].fillna(median)

    if "AssociationFee" in train_df.columns:
        train_df["AssociationFee"] = train_df["AssociationFee"].fillna(0)
        test_df["AssociationFee"] = test_df["AssociationFee"].fillna(0)

    for col in ["GarageSpaces", "ParkingTotal"]:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna(0)
            test_df[col] = test_df[col].fillna(0)

    # ----------------------------
    # Boolean encoding
    # ----------------------------

    train_df[bool_col] = train_df[bool_col].astype(int)
    test_df[bool_col] = test_df[bool_col].astype(int)

    # ----------------------------
    # MultiLabel Encoding
    # ----------------------------

    multi_cols = ["Flooring", "Levels"]

    for col in multi_cols:
        train_df[col] = train_df[col].fillna("").str.split(",")
        test_df[col] = test_df[col].fillna("").str.split(",")

        mlb = MultiLabelBinarizer()

        train_encoded = pd.DataFrame(
            mlb.fit_transform(train_df[col]),
            columns=[f"{col}_{c}" for c in mlb.classes_],
            index=train_df.index
        )

        test_encoded = pd.DataFrame(
            mlb.transform(test_df[col]),
            columns=[f"{col}_{c}" for c in mlb.classes_],
            index=test_df.index
        )

        train_df = train_df.drop(columns=col).join(train_encoded)
        test_df = test_df.drop(columns=col).join(test_encoded)

    # ----------------------------
    # Ordinal Encoding
    # ----------------------------

    mapping = {
        "Unknown": 0,
        "Monthly": 1,
        "Quarterly": 2,
        "SemiAnnually": 3,
        "Annually": 4
    }

    train_df["AssociationFeeFrequency"] = train_df["AssociationFeeFrequency"].map(mapping)
    test_df["AssociationFeeFrequency"] = test_df["AssociationFeeFrequency"].map(mapping)

    # ----------------------------
    # One-Hot Encoding
    # ----------------------------

    one_hot = [
        "CountyOrParish",
        "StateOrProvince",
        "City",
        "PropertyType",
        "PropertySubType",
        'UnifiedDistrict'
    ]

    for col in one_hot:
        top = train_df[col].value_counts().head(200).index

        train_df[col] = train_df[col].where(train_df[col].isin(top), "Other")
        test_df[col] = test_df[col].where(test_df[col].isin(top), "Other")

    train_df = pd.get_dummies(train_df, columns=one_hot, drop_first=True)
    test_df = pd.get_dummies(test_df, columns=one_hot, drop_first=True)

    # Ensure same columns
    train_df, test_df = train_df.align(test_df, join="left", axis=1, fill_value=0)

    # ----------------------------
    # Standardization
    # ----------------------------

    scale = [
        "LivingArea",
        "LotSizeSquareFeet",
        "AssociationFee",
        "DaysOnMarket"
    ]

    scale = [c for c in scale if c in train_df.columns]

    scaler = StandardScaler()
    train_df[scale] = scaler.fit_transform(train_df[scale])
    test_df[scale] = scaler.transform(test_df[scale])

    train_df = train_df.drop(columns=cat_col, errors='ignore')
    test_df = test_df.drop(columns=cat_col, errors='ignore')

    train_df = train_df.drop_duplicates()
    test_df = test_df.drop_duplicates()


    return train_df, test_df

In [30]:
# # Convert CloseDate to datetime
model_df['CloseDate'] = pd.to_datetime(model_df['CloseDate'])
model_df = model_df.sort_values('CloseDate').reset_index(drop=True)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=5),
    "Random Forest": RandomForestRegressor(n_estimators=100)
}
model_results = []

# # Split dataframe into train and test sets using given window
def get_time_split(data, X_months):
    latest_date = data['CloseDate'].max()
    test_start_date = latest_date - pd.DateOffset(months=1)
    train_start_date = test_start_date - pd.DateOffset(months=X_months)
    test_set = data[data['CloseDate'] >= test_start_date]
    train_set = data[(data['CloseDate'] >= train_start_date) & (data['CloseDate'] < test_start_date)]
    return train_set, test_set



train, test = get_time_split(model_df, X_months=12)
train_df, test_df = preproc_df(train, test)

features = train_df.shape[1]
X_train = train_df.drop(columns=['ClosePrice', 'DaysOnMarket'])
y_train = train_df[target]
X_test = test_df.drop(columns=['ClosePrice', 'DaysOnMarket'])
y_test = test_df[target]

for model_name, model in models.items():

    model.fit(X_train, y_train)

    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    train_r2 = r2_score(y_train, train_preds)
    test_r2 = r2_score(y_test, test_preds)

    mae = mean_absolute_error(y_test, test_preds)
    rmse = np.sqrt(mean_squared_error(y_test,test_preds))

    model_results.append({
        "model": model_name,
        "train_r2": train_r2,
        "test_r2": test_r2,
        "mae": mae,
        "rmse": rmse
    })

comparison = pd.DataFrame(model_results)
comparison.sort_values(
by="test_r2",
ascending=False
)
comparison


,model,train_r2,test_r2,mae,rmse
0,Linear Regression,0.826593,0.842103,151764.863540,232041.764667
1,Decision Tree,0.658614,0.658217,230977.811509,341392.249295
2,Random Forest,0.986960,0.914095,97308.348102,171153.928019


In [31]:
print(features)

497


The original dataframe contained 292 features and resulted in the following model performances: 
|    | model             |   train_r2 |    test_r2 |    mae |   rmse |
|---:|:------------------|-----------:|-----------:|-------:|-------:|
|  0 | Linear Regression |   0.800537 |  0.305265  | 424076 | 486730 |
|  1 | Decision Tree     |   0.657183 |  0.65405   | 231739 | 343467 |
|  2 | Random Forest     |   0.986929 |  0.911546  |  98158 | 173675 |


The new dataframe with engineered features and the district data has 489 columns and resulted in the following model performances:
|    | model             |   train_r2 |   test_r2 |      mae |   rmse |
|---:|:------------------|-----------:|----------:|---------:|-------:|
|  0 | Linear Regression |   0.84048  |  0.840729 | 160678   | 255465 |
|  1 | Decision Tree     |   0.661462 |  0.658217 | 230582   | 339728 |
|  2 | Random Forest     |   0.987158 |  0.914095 |  97116.5 | 170947 |

The random forest model didn't improve, the decision tree model improved marginally, and the linear regression model improved by a large amount